# 🧠 Customer Feedback Analyzer
**NLP + LLM Pipeline | Google Colab**

**Features:** Sentiment Analysis (multilingual + Amharic) · Topic Classification · Key Phrase Extraction · Auto Summarization · REST API (Flask + ngrok)

> ⚠️ After Cell 1, go to **Runtime → Restart session**, then run from Cell 2 onward.

## 1. Install Dependencies

*Run once, then restart the runtime.*

In [1]:
!pip install -q \
    "transformers==4.44.2" \
    "sentence-transformers==3.0.1" \
    torch sentencepiece \
    flask flask-cors pyngrok \
    langdetect keybert \
    deep-translator langid \
    openai anthropic python-dotenv \
    pandas numpy scikit-learn requests

print('✅ All dependencies installed — restart runtime now.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 44.2 MB/s eta 0:00:00
✅ All dependencies installed — restart runtime now.


## 2. Imports & Device Setup

In [2]:
import warnings, time, os, socket, threading
warnings.filterwarnings('ignore')

import torch
from typing import Dict, List
from collections import Counter

from transformers import pipeline as hf_pipeline
from keybert import KeyBERT
from deep_translator import GoogleTranslator
import langid

# ── Device ────────────────────────────────────────────────────────────────
DEVICE = 0 if torch.cuda.is_available() else -1
print(f'🖥️  Device: {"GPU" if DEVICE == 0 else "CPU"}')

🖥️  Device: GPU


## 3. Load Models

*Models are loaded once here and reused everywhere — no redundant loading.*

In [3]:
print('📥 Loading multilingual sentiment model (XLM-RoBERTa)...')
# XLM-RoBERTa supports 100+ languages including Amharic
sentiment_pipeline = hf_pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    device=DEVICE,
    truncation=True,
    max_length=512
)

print('📥 Loading zero-shot classifier...')
classifier = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=DEVICE
)

print('📥 Loading summarizer...')
summarizer = hf_pipeline(
    'summarization',
    model='facebook/bart-large-cnn',
    device=DEVICE,
    truncation=True
)

print('📥 Loading KeyBERT (multilingual)...')
kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')

print('✅ All models loaded!')

📥 Loading multilingual sentiment model (XLM-RoBERTa)...


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All models loaded!


## 4. Constants

In [4]:
TOPIC_LABELS = {
    'product_review':  ['quality', 'price', 'delivery', 'packaging', 'functionality',
                        'customer service', 'durability', 'design', 'value for money'],
    'support_ticket':  ['billing issue', 'technical problem', 'account access',
                        'feature request', 'bug report', 'refund request',
                        'shipping issue', 'cancellation'],
    'survey_response': ['user experience', 'product satisfaction', 'support quality',
                        'pricing feedback', 'feature suggestions', 'overall satisfaction'],
    'social_media':    ['brand reputation', 'product complaint', 'praise',
                        'comparison', 'question', 'misinformation', 'viral content'],
    'general':         ['positive experience', 'negative experience', 'neutral feedback',
                        'suggestion', 'complaint', 'compliment', 'question'],
}

SENTIMENT_EMOJI = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}

## 5. Language Helpers

In [5]:
def detect_language(text: str) -> str:
    """Detect the ISO language code of *text* using langid."""
    try:
        lang, _ = langid.classify(text)
        return lang
    except Exception:
        return 'unknown'


def maybe_translate(text: str, lang: str) -> str:
    """Return English translation when *lang* is Amharic, else return *text* unchanged.

    Centralises the translate-if-Amharic logic so every analysis function
    calls this once instead of duplicating the if-block.
    """
    if lang != 'am':
        return text
    try:
        return GoogleTranslator(source='am', target='en').translate(text)
    except Exception as e:
        print(f'⚠️  Translation error: {e}')
        return text

## 6. Core Analysis Functions

In [6]:
def analyze_sentiment(text: str, lang: str = None) -> Dict:
    """Multilingual sentiment analysis. Translates Amharic automatically."""
    if lang is None:
        lang = detect_language(text)
    processing_text = maybe_translate(text, lang)

    result = sentiment_pipeline(processing_text[:512])[0]
    label  = result['label'].lower()
    if   'pos' in label: label = 'positive'
    elif 'neg' in label: label = 'negative'
    else:                label = 'neutral'

    return {
        'sentiment':          label,
        'confidence':         round(result['score'], 4),
        'emoji':              SENTIMENT_EMOJI[label],
        'language':           lang,
        'amharic_translated': lang == 'am',
    }


def classify_topic(text: str, feedback_type: str = 'general', lang: str = None) -> Dict:
    """Zero-shot topic classification against feedback-type-specific label sets."""
    if lang is None:
        lang = detect_language(text)
    processing_text = maybe_translate(text, lang)

    labels = TOPIC_LABELS.get(feedback_type, TOPIC_LABELS['general'])
    result = classifier(processing_text[:512], candidate_labels=labels)

    top3 = [
        {'topic': lbl, 'score': round(score, 4)}
        for lbl, score in zip(result['labels'][:3], result['scores'][:3])
    ]
    return {
        'primary_topic': result['labels'][0],
        'confidence':    round(result['scores'][0], 4),
        'top_topics':    top3,
        'feedback_type': feedback_type,
    }


def extract_keyphrases(text: str, lang: str = None, top_n: int = 8) -> Dict:
    """Key phrase extraction via KeyBERT with MMR diversity."""
    if lang is None:
        lang = detect_language(text)
    processing_text = maybe_translate(text, lang)

    if len(processing_text.split()) < 3:
        return {'keyphrases': [], 'note': 'Text too short for extraction'}

    keywords = kw_model.extract_keywords(
        processing_text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        use_mmr=True,
        diversity=0.6,
        top_n=top_n,
    )
    return {
        'keyphrases': [{'phrase': kw, 'relevance': round(score, 4)} for kw, score in keywords]
    }


def summarize_feedback(text: str, lang: str = None) -> Dict:
    """Abstractive summarisation; skips texts under 30 words."""
    if lang is None:
        lang = detect_language(text)
    processing_text = maybe_translate(text, lang)

    word_count = len(processing_text.split())
    if word_count < 30:
        return {'summary': processing_text, 'note': 'Text too short to summarize'}

    max_len = min(130, max(30, word_count // 3))
    min_len = min(25, max_len - 5)
    result  = summarizer(processing_text[:1024], max_length=max_len, min_length=min_len, do_sample=False)
    return {'summary': result[0]['summary_text']}

## 7. Full Analysis Pipeline

In [7]:
def analyze_feedback(text: str, feedback_type: str = 'general', source: str = None) -> Dict:
    """Run all four analyses on a single piece of feedback and return a unified result."""
    text = text.strip()
    if not text:
        return {'error': 'Empty feedback text'}

    start = time.time()
    lang  = detect_language(text)          # detect once, pass everywhere

    return {
        'input': {
            'text':          text,
            'feedback_type': feedback_type,
            'source':        source or 'unknown',
            'language':      lang,
            'word_count':    len(text.split()),
        },
        'sentiment':  analyze_sentiment(text, lang),
        'topic':      classify_topic(text, feedback_type, lang),
        'keyphrases': extract_keyphrases(text, lang),
        'summary':    summarize_feedback(text, lang),
        'meta': {
            'processing_time_sec': round(time.time() - start, 2),
            'model_device':        'GPU' if DEVICE == 0 else 'CPU',
        },
    }


def batch_analyze(feedbacks: List[Dict]) -> List[Dict]:
    """Analyze a list of feedback dicts, logging progress."""
    results = []
    total   = len(feedbacks)
    for i, item in enumerate(feedbacks, 1):
        print(f'  Processing {i}/{total}...')
        results.append(analyze_feedback(
            text=item.get('text', ''),
            feedback_type=item.get('feedback_type', 'general'),
            source=item.get('source', 'unknown'),
        ))
    return results


def generate_insights(results: List[Dict]) -> Dict:
    """Aggregate sentiment/topic/keyphrase statistics across a batch result list."""
    if not results:
        return {}

    sentiments  = [r['sentiment']['sentiment'] for r in results if 'sentiment' in r]
    topics      = [r['topic']['primary_topic']  for r in results if 'topic'    in r]
    all_phrases = [
        kp['phrase']
        for r in results if 'keyphrases' in r
        for kp in r['keyphrases'].get('keyphrases', [])
    ]

    sentiment_counts = dict(Counter(sentiments))
    total   = len(sentiments)
    pos_pct = round(sentiment_counts.get('positive', 0) / total * 100, 1) if total else 0
    neg_pct = round(sentiment_counts.get('negative', 0) / total * 100, 1) if total else 0

    return {
        'total_feedbacks':        total,
        'sentiment_distribution': sentiment_counts,
        'positive_rate_pct':      pos_pct,
        'negative_rate_pct':      neg_pct,
        'top_topics':             dict(Counter(topics).most_common(5)),
        'top_keyphrases':         dict(Counter(all_phrases).most_common(10)),
        'alert': 'High negative feedback detected!' if neg_pct > 40 else None,
    }


print('✅ Pipeline ready!')

✅ Pipeline ready!


## 8. Quick Demo

In [8]:
def _print_result(label: str, r: Dict) -> None:
    print(f'\n--- {label} ---')
    print(f"Sentiment : {r['sentiment']['sentiment']} {r['sentiment']['emoji']} ({r['sentiment']['confidence']:.2%})")
    print(f"Topic     : {r['topic']['primary_topic']}")
    print(f"Phrases   : {[kp['phrase'] for kp in r['keyphrases']['keyphrases'][:4]]}")
    if r['summary'].get('summary'):
        print(f"Summary   : {r['summary']['summary']}")


_print_result('English Product Review', analyze_feedback(
    text='This product is absolutely amazing! Great quality, fast delivery, and excellent customer support.',
    feedback_type='product_review',
    source='amazon',
))

_print_result('Amharic Product Review', analyze_feedback(
    text='ይህ ምርት በጣም ጥሩ ነው። ፈጣን ማድረስ እና ጥሩ ጥራት አለው።',
    feedback_type='product_review',
    source='social_media',
))


--- English Product Review ---
Sentiment : positive 😊 (93.46%)
Topic     : quality
Phrases   : ['excellent customer', 'delivery excellent', 'great quality', 'absolutely amazing']
Summary   : This product is absolutely amazing! Great quality, fast delivery, and excellent customer support.

--- Amharic Product Review ---
Sentiment : positive 😊 (92.05%)
Topic     : quality
Phrases   : ['product great', 'delivery good', 'good quality', 'fast delivery']
Summary   : This product is great. Fast delivery and good quality.


## 9. Flask REST API

*All routes defined here. The server is started in the next cell.*

In [9]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


def _require_json_field(field: str):
    """Helper: parse request JSON and return (data, error_response)."""
    data = request.get_json(force=True, silent=True)
    if not data or field not in data:
        return None, (jsonify({'error': f'Missing required field: {field}'}), 400)
    return data, None


@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'device': 'GPU' if DEVICE == 0 else 'CPU'})


@app.route('/api/analyze', methods=['POST'])
def api_analyze():
    data, err = _require_json_field('text')
    if err: return err
    try:
        return jsonify(analyze_feedback(
            text=data['text'],
            feedback_type=data.get('feedback_type', 'general'),
            source=data.get('source'),
        ))
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/api/analyze/batch', methods=['POST'])
def api_batch_analyze():
    data, err = _require_json_field('feedbacks')
    if err: return err
    try:
        feedbacks = data['feedbacks']
        if len(feedbacks) > 100:
            return jsonify({'error': 'Max batch size is 100'}), 400
        results  = batch_analyze(feedbacks)
        insights = generate_insights(results)
        return jsonify({'results': results, 'insights': insights, 'count': len(results)})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/api/sentiment', methods=['POST'])
def api_sentiment():
    data, err = _require_json_field('text')
    if err: return err
    try:    return jsonify(analyze_sentiment(data['text']))
    except Exception as e: return jsonify({'error': str(e)}), 500


@app.route('/api/topics', methods=['POST'])
def api_topics():
    data, err = _require_json_field('text')
    if err: return err
    try:    return jsonify(classify_topic(data['text'], data.get('feedback_type', 'general')))
    except Exception as e: return jsonify({'error': str(e)}), 500


@app.route('/api/keyphrases', methods=['POST'])
def api_keyphrases():
    data, err = _require_json_field('text')
    if err: return err
    try:    return jsonify(extract_keyphrases(data['text'], top_n=data.get('top_n', 8)))
    except Exception as e: return jsonify({'error': str(e)}), 500


@app.route('/api/summarize', methods=['POST'])
def api_summarize():
    data, err = _require_json_field('text')
    if err: return err
    try:    return jsonify(summarize_feedback(data['text']))
    except Exception as e: return jsonify({'error': str(e)}), 500


@app.route('/api/insights', methods=['POST'])
def api_insights():
    data, err = _require_json_field('feedbacks')
    if err: return err
    try:
        results  = batch_analyze(data['feedbacks'])
        insights = generate_insights(results)
        return jsonify(insights)
    except Exception as e: return jsonify({'error': str(e)}), 500


print('✅ Flask routes registered!')

✅ Flask routes registered!


## 10. Start Server & Expose via ngrok

⚠️ Paste your ngrok token below before running.

In [10]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = '2repFCx5APaDv49xfYdw7JhY9a8_6cHiHvkjMycB38vqbLLyN'  # ← replace with your token

# ── Release port if occupied ──────────────────────────────────────────────
os.system('fuser -k 5000/tcp 2>/dev/null')
time.sleep(1)

# ── Pick a free port ──────────────────────────────────────────────────────
def _free_port(preferred: int = 5000) -> int:
    for port in [preferred, 0]:
        try:
            with socket.socket() as s:
                s.bind(('', port))
                return s.getsockname()[1]
        except OSError:
            continue
    raise RuntimeError('No free port found')

PORT = _free_port()
print(f'🔌 Using port: {PORT}')

# ── Start Flask in background ─────────────────────────────────────────────
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False),
    daemon=True,
).start()
time.sleep(2)

# ── Open ngrok tunnel ─────────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
time.sleep(1)
public_url = ngrok.connect(PORT).public_url

print(f'\n🚀 NLP Server is LIVE!')
print(f'📡 Public URL : {public_url}')
print(f'\n📋 Spring Boot application.properties:')
print(f'   nlp.service.base-url={public_url}')
print(f'\n🔗 Endpoints:')
for path in ['/api/analyze', '/api/analyze/batch', '/api/sentiment',
             '/api/topics', '/api/keyphrases', '/api/summarize',
             '/api/insights', '/health']:
    method = 'GET' if path == '/health' else 'POST'
    print(f'   {method:4s} {public_url}{path}')

🔌 Using port: 5000
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



🚀 NLP Server is LIVE!
📡 Public URL : https://c04a-8-229-130-155.ngrok-free.app

📋 Spring Boot application.properties:
   nlp.service.base-url=https://c04a-8-229-130-155.ngrok-free.app

🔗 Endpoints:
   POST https://c04a-8-229-130-155.ngrok-free.app/api/analyze
   POST https://c04a-8-229-130-155.ngrok-free.app/api/analyze/batch
   POST https://c04a-8-229-130-155.ngrok-free.app/api/sentiment
   POST https://c04a-8-229-130-155.ngrok-free.app/api/topics
   POST https://c04a-8-229-130-155.ngrok-free.app/api/keyphrases
   POST https://c04a-8-229-130-155.ngrok-free.app/api/summarize
   POST https://c04a-8-229-130-155.ngrok-free.app/api/insights
   GET  https://c04a-8-229-130-155.ngrok-free.app/health


## 11. Sample Test Calls

In [11]:
import requests

BASE = 'http://localhost:5000'  # or swap in the ngrok public_url


def safe_post(endpoint: str, payload: dict) -> dict | None:
    """POST *payload* to *endpoint* and return parsed JSON, or None on failure."""
    try:
        r = requests.post(f'{BASE}{endpoint}', json=payload, timeout=60)
        r.raise_for_status()
        return r.json()
    except requests.HTTPError as e:
        print(f'❌ HTTP {e.response.status_code}: {e.response.text}')
    except Exception as e:
        print(f'❌ Request error: {e}')
    return None


# ── Test 1: Full English review ───────────────────────────────────────────
result = safe_post('/api/analyze', {
    'text': 'The delivery was delayed by 2 weeks and the package arrived damaged. Very disappointed.',
    'feedback_type': 'product_review',
    'source': 'website',
})
if result:
    print('=== Full Analysis ===')
    print(f"Sentiment  : {result['sentiment']['sentiment']} {result['sentiment']['emoji']}")
    print(f"Topic      : {result['topic']['primary_topic']}")
    print(f"Key Phrases: {[k['phrase'] for k in result['keyphrases']['keyphrases'][:3]]}")
    print(f"Summary    : {result['summary']['summary']}")


# ── Test 2: Amharic support ticket ────────────────────────────────────────
result2 = safe_post('/api/analyze', {
    'text': 'ለምን ደሞዜን አትከፍሉኝም?',
    'feedback_type': 'support_ticket',
    'source': 'email',
})
if result2:
    print('\n=== Amharic Support Ticket ===')
    print(f"Language   : {result2['input']['language']}")
    print(f"Sentiment  : {result2['sentiment']['sentiment']} {result2['sentiment']['emoji']}")
    print(f"Topic      : {result2['topic']['primary_topic']}")

INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 07:03:58] "POST /api/analyze HTTP/1.1" 200 -


=== Full Analysis ===
Sentiment  : negative 😞
Topic      : delivery
Key Phrases: ['delivery delayed', 'delayed weeks', 'package arrived']
Summary    : The delivery was delayed by 2 weeks and the package arrived damaged. Very disappointed.


INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 07:03:58] "POST /api/analyze HTTP/1.1" 200 -



=== Amharic Support Ticket ===
Language   : am
Sentiment  : negative 😞
Topic      : billing issue
